# Indikatoren-Analyse für ausgewählte Länder

Analyse der Differenzen (2015-2023) für:
- Portugal (PRT)
- Vanuatu (VUT)
- Botswana (BWA)
- Somalia (SOM)

In [ ]:
import pandas as pd
import os
from pathlib import Path
import numpy as np

## 1. Einlesen aller Indikatoren-Dateien

In [ ]:
# Pfad zum indicators-Verzeichnis
indicators_path = Path("data/vuln_indicators")

# Länder die analysiert werden sollen
countries = {
    'PRT': 'Portugal',
    'VUT': 'Vanuatu',
    'BWA': 'Botswana',
    'SOM': 'Somalia'
}

# Dictionary zum Speichern aller DataFrames
indicator_data = {}

# Erlaubte Präfixe für Vulnerability-Indikatoren
allowed_prefixes = ['id_food_', 'id_wate_', 'id_heal_', 'id_ecos_', 'id_habi_', 'id_infr_']

# Alle Ordner finden, die mit "id_" beginnen (vor Filter)
all_id_folders = [f for f in indicators_path.iterdir() if f.is_dir() and f.name.startswith('id_')]

# Nur Vulnerability-Indikatoren filtern
id_folders = sorted([f for f in all_id_folders if any(f.name.startswith(prefix) for prefix in allowed_prefixes)])

print(f"Alle id_-Ordner gefunden: {len(all_id_folders)}")
print(f"Vulnerability-Indikatoren (nach Filter): {len(id_folders)}")

# Alle score.csv Dateien einlesen
for folder in id_folders:
    input_file = folder / "score.csv"
    if input_file.exists():
        try:
            df = pd.read_csv(input_file)
            indicator_name = folder.name
            indicator_data[indicator_name] = df
            print(f"✓ {indicator_name}: {len(df)} Länder eingelesen")
        except Exception as e:
            print(f"✗ Fehler bei {folder.name}: {e}")

print(f"\nInsgesamt {len(indicator_data)} Vulnerability-Indikatoren erfolgreich eingelesen")

## 2. Berechnung der Differenzen (2015-2023)

In [ ]:
# Dictionary für die Ergebnisse
results = {country_code: {} for country_code in countries.keys()}

# Für jeden Indikator die Differenzen berechnen
for indicator_name, df in indicator_data.items():
    # Prüfen ob die Jahre 2015 und 2023 als Spalten existieren
    if '2015' in df.columns and '2023' in df.columns:
        for country_code in countries.keys():
            # Land im DataFrame finden
            country_row = df[df['ISO3'] == country_code]
            
            if not country_row.empty:
                try:
                    value_2015 = pd.to_numeric(country_row['2015'].values[0], errors='coerce')
                    value_2023 = pd.to_numeric(country_row['2023'].values[0], errors='coerce')
                    
                    # Differenz berechnen (nur wenn beide Werte verfügbar sind)
                    if pd.notna(value_2015) and pd.notna(value_2023):
                        diff = value_2023 - value_2015
                        results[country_code][indicator_name] = {
                            '2015': value_2015,
                            '2023': value_2023,
                            'Differenz': diff
                        }
                except Exception as e:
                    pass  # Fehler überspringen

# Überblick über die Ergebnisse
for country_code in countries.keys():
    print(f"{countries[country_code]} ({country_code}): {len(results[country_code])} Indikatoren mit gültigen Daten")

## 3. Ranglisten erstellen

In [ ]:
# Für jedes Land eine Rangliste nach absoluter Differenz erstellen
for country_code, country_name in countries.items():
    print(f"\n{'='*80}")
    print(f"RANGLISTE FÜR {country_name.upper()} ({country_code})")
    print(f"{'='*80}\n")
    
    # DataFrame aus den Ergebnissen erstellen
    country_results = results[country_code]
    
    if country_results:
        df_ranking = pd.DataFrame.from_dict(country_results, orient='index')
        
        # Nach absoluter Differenz sortieren (größte Veränderung zuerst)
        df_ranking['Abs_Differenz'] = df_ranking['Differenz'].abs()
        df_ranking = df_ranking.sort_values('Abs_Differenz', ascending=False)
        
        # Formatierung für bessere Lesbarkeit
        df_ranking['2015'] = df_ranking['2015'].round(4)
        df_ranking['2023'] = df_ranking['2023'].round(4)
        df_ranking['Differenz'] = df_ranking['Differenz'].round(4)
        df_ranking['Abs_Differenz'] = df_ranking['Abs_Differenz'].round(4)
        
        # Top 10 anzeigen
        print("Top 10 Indikatoren mit größter absoluter Veränderung:")
        print("-" * 80)
        display(df_ranking[['2015', '2023', 'Differenz', 'Abs_Differenz']].head(10))
        
        print(f"\n\nGesamtanzahl Indikatoren mit Daten: {len(df_ranking)}")
    else:
        print(f"Keine Daten verfügbar für {country_name}")

## 4. Detaillierte Ranglisten nach positiven und negativen Veränderungen

In [ ]:
# Zusätzliche Analyse: Positive vs. Negative Veränderungen
for country_code, country_name in countries.items():
    print(f"\n{'='*80}")
    print(f"DETAILANALYSE FÜR {country_name.upper()} ({country_code})")
    print(f"{'='*80}\n")
    
    country_results = results[country_code]
    
    if country_results:
        df_detail = pd.DataFrame.from_dict(country_results, orient='index')
        
        # Positive und negative Veränderungen trennen
        df_positive = df_detail[df_detail['Differenz'] > 0].sort_values('Differenz', ascending=False)
        df_negative = df_detail[df_detail['Differenz'] < 0].sort_values('Differenz', ascending=True)
        
        print(f"📈 TOP 5 GRÖSSTE POSITIVE VERÄNDERUNGEN (Anstieg):")
        print("-" * 80)
        if len(df_positive) > 0:
            display(df_positive[['2015', '2023', 'Differenz']].head(5))
        else:
            print("Keine positiven Veränderungen gefunden")
        
        print(f"\n📉 TOP 5 GRÖSSTE NEGATIVE VERÄNDERUNGEN (Rückgang):")
        print("-" * 80)
        if len(df_negative) > 0:
            display(df_negative[['2015', '2023', 'Differenz']].head(5))
        else:
            print("Keine negativen Veränderungen gefunden")
        
        print(f"\n📊 Zusammenfassung:")
        print(f"   • Positive Veränderungen: {len(df_positive)}")
        print(f"   • Negative Veränderungen: {len(df_negative)}")
        print(f"   • Keine Veränderung: {len(df_detail[df_detail['Differenz'] == 0])}")
    else:
        print(f"Keine Daten verfügbar für {country_name}")